# "THE PRICE IS RIGHT" - Dự án Capstone

Notebook này xây dựng một mô hình có thể ước lượng giá của một sản phẩm dựa trên mô tả bằng văn bản, sử dụng dữ liệu đã được crawl từ Amazon.

Mục tiêu chính: từ mô tả sản phẩm (summary) suy ra khoảng giá gần với giá thật nhất có thể.

## Quy trình chính

- Ngày 1: Thu thập và chuẩn bị dữ liệu
- Ngày 2: Tiền xử lý dữ liệu
- Ngày 3: Đánh giá, baseline và ML truyền thống
- Ngày 4: Deep Learning và LLM
- Ngày 5: Fine-tuning mô hình frontier

## NGÀY 4: Mạng nơ-ron và LLM

Hôm nay, ta sẽ đi từ các mô hình học máy truyền thống, sang mạng nơ-ron, rồi đến các mô hình ngôn ngữ lớn (LLM). Đây là bước chuyển tiếp quan trọng trong quá trình xây dựng hệ thống dự đoán giá sản phẩm.

> Notebook này giúp người học hiểu cách dữ liệu văn bản được chuyển thành vector, sau đó được đưa vào mạng nơ-ron, và cuối cùng so sánh với khả năng suy luận của các mô hình LLM hiện đại.

In [ ]:
# imports

import os
from dotenv import load_dotenv
from huggingface_hub import login
from pricer.evaluator import evaluate
from litellm import completion
from pricer.items import Item
import numpy as np
from tqdm.notebook import tqdm
import csv
from sklearn.feature_extraction.text import HashingVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from torch.optim.lr_scheduler import CosineAnnealingLR


In [ ]:
LITE_MODE = False

load_dotenv(override=True)
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

# Trước khi xem xét Mạng nơ-ron nhân tạo

## Có một kiểu mạng nơ-ron khác mà ta có thể cân nhắc

Ở phần này, chúng ta đang đặt nền tảng cho việc hiểu rằng không phải mọi bài toán đều cần mô hình học máy truyền thống. Một mô hình nơ-ron có thể học trực tiếp từ dữ liệu dạng văn bản và lượng hóa mối quan hệ giữa mô tả với giá thực tế.

Một cách đơn giản: thay vì dựa hoàn toàn vào quy tắc thủ công, ta cho mô hình học biểu diễn từ dữ liệu. Đây là tiền đề để tiếp cận mạng nơ-ron và LLM.

> Cột mốc quan trọng: dữ liệu mô tả sản phẩm không chỉ là chuỗi ký tự, mà có thể được biến đổi thành vector rồi đưa vào mạng học để dự đoán giá.

In [ ]:
# Write the test set to a CSV

with open('human_in.csv', 'w', encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    for t in test[:100]:
        writer.writerow([t.summary, 0])

In [ ]:
# Read it back in

human_predictions = []
with open('human_out.csv', 'r', encoding="utf-8") as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        human_predictions.append(float(row[1]))

In [ ]:
def human_pricer(item):
    idx = test.index(item)
    return human_predictions[idx]

In [ ]:
human = human_pricer(test[0])
actual = test[0].price
print(f"Human predicted {human} for an item that actually costs {actual}")


In [ ]:
evaluate(human_pricer, test, size=100)

# Bây giờ là một mạng nơ-ron cơ bản

Trong phần còn lại của khóa học, ta sẽ đi sâu hơn vào cách mạng nơ-ron hoạt động và cách huấn luyện chúng.

Đây chỉ là một bản xem qua nhanh, không phải kiến thức đầy đủ về mạng nơ-ron ngay lúc này.

Ta sẽ tự xây dựng một Mạng nơ-ron từ đầu, bằng PyTorch, để tạo cảm giác trực quan về cách nó học.

Mục tiêu là giúp người học hiểu ý tưởng chung trước khi đi vào các mô hình phức tạp hơn.

> Cell này đặt nền móng cho khái niệm: dữ liệu văn bản -> vector -> mạng nơ-ron -> giá dự đoán.

In [ ]:
# Chuẩn bị tài liệu và giá sản phẩm

# Tạo mảng giá trị mục tiêu y từ dữ liệu huấn luyện
# Mỗi item trong tập train có thuộc tính price (giá) và summary (mô tả ngắn gọn)
y = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

# Mỗi văn bản mô tả sản phẩm sẽ được dùng làm đầu vào để dự đoán giá.
# Dữ liệu này tương đương với cặp (mô tả sản phẩm, giá thật).

In [ ]:
# Dùng HashingVectorizer để tạo Bag of Words
# Với binary=True, mỗi từ trong văn bản chỉ được mã hóa thành 1 hoặc 0 (có/không xuất hiện),
# thay vì đếm số lần lặp lại như trong CountVectorizer truyền thống.

np.random.seed(42)
vectorizer = HashingVectorizer(n_features=5000, stop_words='english', binary=True)
X = vectorizer.fit_transform(documents)

# Kết quả: mỗi mô tả sản phẩm được chuyển thành vector số học có độ dài cố định 5000.
# Đây là dạng dữ liệu mà mạng nơ-ron có thể đọc được.

In [ ]:
# Khai báo mạng nơ-ron - đây là một mạng 8 lớp bằng PyTorch

class NeuralNetwork(nn.Module):
    def __init__(self, input_size):
        super(NeuralNetwork, self).__init__()
        # Mỗi lớp học sẽ biến đổi vector đầu vào thành không gian ẩn mới.
        self.layer1 = nn.Linear(input_size, 128)
        self.layer2 = nn.Linear(128, 64)
        self.layer3 = nn.Linear(64, 64)
        self.layer4 = nn.Linear(64, 64)
        self.layer5 = nn.Linear(64, 64)
        self.layer6 = nn.Linear(64, 64)
        self.layer7 = nn.Linear(64, 64)
        self.layer8 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        # Propagation xuôi: chuyển dữ liệu qua từng lớp và áp dụng ReLU
        output1 = self.relu(self.layer1(x))
        output2 = self.relu(self.layer2(output1))
        output3 = self.relu(self.layer3(output2))
        output4 = self.relu(self.layer4(output3))
        output5 = self.relu(self.layer5(output4))
        output6 = self.relu(self.layer6(output5))
        output7 = self.relu(self.layer7(output6))
        output8 = self.layer8(output7)
        return output8

# Đây là một mạng nơ-ron đơn giản, đủ để tạo cảm giác trực quan về cách neural network học.
# Không cần phải hiểu từng chi tiết ngay lúc này; quan trọng là nắm được ý tưởng: dữ liệu đi qua nhiều lớp, từng lớp trích xuất đặc trưng hơn.

In [ ]:
# Chuyển dữ liệu thành tensor của PyTorch
# X_train_tensor: ma trận đặc trưng của dữ liệu huấn luyện
# y_train_tensor: vector giá mục tiêu, thêm chiều 1 vì mạng đầu ra là 1 giá trị
X_train_tensor = torch.FloatTensor(X.toarray())
y_train_tensor = torch.FloatTensor(y).unsqueeze(1)

# Chia dữ liệu thành tập train và validation
X_train, X_val, y_train, y_val = train_test_split(X_train_tensor, y_train_tensor, test_size=0.01, random_state=42)

# Tạo DataLoader để đưa dữ liệu vào theo từng batch
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

# Khởi tạo mô hình
input_size = X_train_tensor.shape[1]
model = NeuralNetwork(input_size)

# Bây giờ model đã sẵn sàng để học từ dữ liệu văn bản đã được mã hóa.

In [ ]:
# Đếm số tham số có thể học của mô hình
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Number of trainable parameters: {trainable_params:,}")

# Đây là số lượng tham số mà mô hình sẽ cập nhật trong quá trình huấn luyện.
# Nếu số này lớn, mô hình có nhiều capacity hơn để học nhưng cũng tốn thời gian và dữ liệu hơn.

In [ ]:
# Định nghĩa hàm mất mát và optimizer
# MSELoss đo sai số giữa giá dự đoán và giá thật
# Adam là optimizer giúp cập nhật trọng số theo hướng giảm lỗi hiệu quả

loss_function = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Chúng ta sẽ huấn luyện 2 vòng lặp hoàn chỉnh qua toàn bộ dữ liệu
EPOCHS = 2

for epoch in range(EPOCHS):
    model.train()
    for batch_X, batch_y in tqdm(train_loader):
        optimizer.zero_grad()

        # 4 bước chính của quá trình huấn luyện:
        # 1. Forward pass: đưa dữ liệu qua mạng nơ-ron
        # 2. Tính loss: so sánh output với nhãn thật
        # 3. Backward pass: tính gradient
        # 4. Optimize: cập nhật trọng số
        outputs = model(batch_X)
        loss = loss_function(outputs, batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val)
        val_loss = loss_function(val_outputs, y_val)

    print(f'Epoch [{epoch+1}/{EPOCHS}], Train Loss: {loss.item():.3f}, Val Loss: {val_loss.item():.3f}')

# Mục tiêu: làm giảm loss trên tập validation, cho thấy mô hình đang học được quy luật từ dữ liệu.

In [ ]:
def neural_network(item):
    model.eval()
    with torch.no_grad():
        # Chuyển mô tả sản phẩm thành vector bằng cùng vectorizer đã train trước đó
        vector = vectorizer.transform([item.summary])
        vector = torch.FloatTensor(vector.toarray())
        result = model(vector)[0].item()
    return max(0, result)

# Hàm này dùng để dự đoán giá cho một sản phẩm mới, dựa trên mô tả của nó.
# Giá âm không hợp lý, nên ta clamp về 0 nếu mô hình dự đoán âm.

In [ ]:
evaluate(neural_network, test)

# Bây giờ là frontier models!

Hãy xem các mô hình frontier hoạt động như thế nào khi không cần huấn luyện lại gì cả; chúng chỉ suy luận dựa trên tri thức có sẵn trong quá trình pretraining.

Ngày mai, ta sẽ thực hiện fine-tuning cho một mô hình frontier.

> Đây là bước quan trọng để hiểu sự khác biệt giữa mô hình học từ dữ liệu bằng cách tối ưu trọng số và mô hình LLM sử dụng khả năng suy luận tổng quát từ dữ liệu lớn đã được huấn luyện trước.

In [ ]:
def messages_for(item):
    # Ta yêu cầu mô hình ước lượng giá của sản phẩm dựa trên mô tả.
    # Mô hình được yêu cầu chỉ trả về số tiền, không giải thích dài dòng.
    message = f"Estimate the price of this product. Respond with the price, no explanation\n\n{item.summary}"
    return [{"role": "user", "content": message}]

# Cell này chuyển thông tin sản phẩm thành prompt chuẩn để gửi tới mô hình LLM.

In [ ]:
print(test[0].summary)

In [ ]:
messages_for(test[0])

In [ ]:
# Hàm dự đoán cho gpt-4.1-nano

def gpt_4__1_nano(item):
    response = completion(model="openai/gpt-4.1-nano", messages=messages_for(item))
    return response.choices[0].message.content

# Mô hình này được gọi để suy đoán giá từ mô tả bằng prompt rất ngắn.
# Đây là ví dụ minh họa cho việc dùng LLM như một "price estimator" mà không cần huấn luyện lại.

In [ ]:
gpt_4__1_nano(test[0])

In [ ]:
test[0].price

In [ ]:
evaluate(gpt_4__1_nano, test)

In [ ]:
def claude_opus_4_5(item):
    response = completion(model="anthropic/claude-opus-4-5", messages=messages_for(item))
    return response.choices[0].message.content

# Claude Opus 4.5 là một model mạnh, được dùng để so sánh với các model khác.
# Ta đánh giá chất lượng ước lượng giá của từng model một cách khách quan qua evaluate().

In [ ]:
evaluate(claude_opus_4_5, test)

In [ ]:
def gemini_3_pro_preview(item):
    response = completion(model="gemini/gemini-3-pro-preview", messages=messages_for(item), reasoning_effort='low')
    return response.choices[0].message.content

# Gemini được thử nghiệm với độ ưu tiên suy luận thấp, phù hợp với bài toán ước lượng giá nhanh và đơn giản.
# Việc so sánh các model khác nhau giúp thấy rõ hiệu suất và độ đáng tin cậy của từng mô hình.

In [ ]:
evaluate(gemini_3_pro_preview, test, size=50, workers=2)

In [ ]:
def gemini_2__5_flash_lite(item):
    response = completion(model="gemini/gemini-3.1-flash-lite", messages=messages_for(item))
    return response.choices[0].message.content

# Model này là lựa chọn nhẹ hơn, phù hợp để test tốc độ và hiệu suất trên dữ liệu cùng loại.
# So sánh các model giúp người học hiểu rằng không phải model nào cũng tốt như nhau đối với từng bài toán.

In [ ]:
evaluate(gemini_2__5_flash_lite, test)

In [ ]:

def grok_4__1_fast(item):
    response = completion(model="xai/grok-4-1-fast-non-reasoning", messages=messages_for(item), seed=42)
    return response.choices[0].message.content

In [ ]:
evaluate(grok_4__1_fast, test)

In [ ]:
# The function for gpt-5.1

def gpt_5__1(item):
    response = completion(model="gpt-5.1", messages=messages_for(item), reasoning_effort='high', seed=42)
    return response.choices[0].message.content


In [ ]:
evaluate(gpt_5__1, test)